<a href="https://colab.research.google.com/github/pushpakgaidhane294/Gen_AI_Lab/blob/main/Practical5_Gen_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required libraries
!pip install -q transformers datasets evaluate accelerate

# 2. Import libraries
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

# 3. Load and Tokenize Dataset
print("Loading dataset and tokenizer...")
dataset = load_dataset("SetFit/ag_news")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. Initialize Model
print("\nInitializing model...")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

# 5. Define Metrics
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6. Setup Training Arguments (Increased dataset to 2,000 entries for actual accuracy)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(2000)),
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(400)),
    compute_metrics=compute_metrics,
)

# 7. Train the Model
print("\nStarting training...")
trainer.train()

# 8. Setup Category Mapping and Classifier Pipeline
custom_labels = {
    "LABEL_0": "Politics (World)",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Technology (Sci/Tech)"
}

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# 9. Test Sample Predictions
print("\n==========================================")
print("--- PREDEFINED SAMPLE PREDICTIONS ---")
print("==========================================\n")

articles = [
    "The new flagship smartphone features a 3nm quantum processor.",
    "The championship game ended with a stunning buzzer-beater.",
    "Diplomatic talks between the two nations stalled over border disputes."
]

for article in articles:
    result = classifier(article)[0]
    predicted_label = custom_labels[result['label']]
    print(f"Article: '{article}'")
    print(f"Category: {predicted_label} | Confidence: {result['score']:.2f}\n")

# 10. Interactive Text Box
print("==========================================")
print("--- INTERACTIVE CUSTOM CLASSIFIER ---")
print("Type any text below to classify (or type 'quit' to stop)")
print("==========================================\n")

while True:
    user_input = input("Enter text: ")

    if user_input.lower().strip() in ['quit', 'exit']:
        print("Stopping interactive mode.")
        break

    if not user_input.strip():
        continue

    result = classifier(user_input)[0]
    predicted_label = custom_labels[result['label']]
    print(f"-> Category: {predicted_label} | Confidence: {result['score']:.2f}\n")

Loading dataset and tokenizer...

Initializing model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.421934,0.431132,0.857500
2,0.277398,0.410203,0.880000
3,0.165722,0.421624,0.875000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- PREDEFINED SAMPLE PREDICTIONS ---

Article: 'The new flagship smartphone features a 3nm quantum processor.'
Category: Technology (Sci/Tech) | Confidence: 0.97

Article: 'The championship game ended with a stunning buzzer-beater.'
Category: Sports | Confidence: 0.97

Article: 'Diplomatic talks between the two nations stalled over border disputes.'
Category: Politics (World) | Confidence: 0.98

--- INTERACTIVE CUSTOM CLASSIFIER ---
Type any text below to classify (or type 'quit' to stop)

Enter text: Rachin Ravindra ruled out of India T20Is; Milne and Bracewell return
-> Category: Sports | Confidence: 0.94

Enter text: The great-power politics of AI: As Trump and Xi meet in Washington, the battle for global AI governance intensifies
-> Category: Technology (Sci/Tech) | Confidence: 0.90

Enter text: 'Majority not with Gyanesh Kumar': Kapil Sibal demands suspension of CEC, halt to SIR process
-> Category: Politics (World) | Confidence: 0.97

Enter text: exit
Stopping interactive mode.